# hls4ml Neural Network — FPGA vs CPU
Transfer this notebook together with the new `design_1_wrapper.bit` / `.hwh` (built with the hls4ml IP + AXI DMA) and the `.npy` weight files.

The hls4ml IP uses **AXI-Stream** — data is pushed through a DMA, not via AXI-Lite pointer registers.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

BITFILE = "/home/xilinx/jupyter_notebooks/neural_network/design_1_wrapper.bit"

print("Loading overlay...", end=" ", flush=True)
t0 = time.time()
ol = Overlay(BITFILE)
print(f"done  [{(time.time()-t0)*1000:.0f} ms]")
print("IPs found:", list(ol.ip_dict.keys()))

In [ ]:
# hls4ml uses AXI-Stream — grab the DMA block
dma = ol.axi_dma_0   # adjust name to match your block design
print("DMA found:", dma)

In [ ]:
print("Loading weights and test sample...", end=" ", flush=True)
W1 = np.load("W1.npy").astype(np.float32)
b1 = np.load("b1.npy").astype(np.float32)
W2 = np.load("W2.npy").astype(np.float32)
b2 = np.load("b2.npy").astype(np.float32)

x_test = np.fromfile("x.bin", dtype=np.float32)          # shape (784,)
label  = int(np.fromfile("y_expected.bin", dtype=np.float32).argmax())
print("done")
print(f"True label: {label}")

In [ ]:
# hls4ml expects ap_fixed<16,6> — scale float input to match
# The VivadoAccelerator wrapper accepts raw float32 over AXI-Stream and converts internally.
# Allocate physically contiguous input/output buffers.
print("Allocating DMA buffers...", end=" ", flush=True)
in_buf  = allocate(shape=(784,), dtype=np.float32)
out_buf = allocate(shape=(10,),  dtype=np.float32)
in_buf[:] = x_test
in_buf.flush()
print("done")

In [ ]:
print("=" * 42)
print("  FPGA INFERENCE (hls4ml)")
print("=" * 42)

print("  [1/3] Starting DMA transfer...", end=" ", flush=True)
fpga_start = time.time()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
fpga_end = time.time()
fpga_time_ms = (fpga_end - fpga_start) * 1000
print(f"done  [{fpga_time_ms:.3f} ms]")

print("  [2/3] Reading output...", end=" ", flush=True)
out_buf.invalidate()
fpga_pred = int(np.argmax(out_buf))
print("done")

print(f"\n  Prediction : {fpga_pred}  ({'correct' if fpga_pred == label else 'WRONG'})")
print(f"  Time       : {fpga_time_ms:.3f} ms")
print(f"  Logits     : {np.array(out_buf).round(3)}")
print("=" * 42)

In [ ]:
print("=" * 42)
print("  CPU INFERENCE")
print("=" * 42)

def forward_numpy(x, W1, b1, W2, b2):
    h = np.dot(W1, x) + b1
    h = np.maximum(h, 0)
    return np.dot(W2, h) + b2

N = 100
print(f"  Running {N} iterations...", end=" ", flush=True)
cpu_start = time.time()
for _ in range(N):
    y_cpu = forward_numpy(x_test, W1, b1, W2, b2)
cpu_end = time.time()
cpu_time_ms = (cpu_end - cpu_start) / N * 1000
cpu_pred = int(np.argmax(y_cpu))
print("done")

print(f"\n  Prediction : {cpu_pred}  ({'correct' if cpu_pred == label else 'WRONG'})")
print(f"  Time/sample: {cpu_time_ms:.3f} ms  (avg of {N} runs)")
print("=" * 42)

In [ ]:
speedup = cpu_time_ms / fpga_time_ms

print()
print("=" * 42)
print("  RESULTS SUMMARY  (hls4ml IP)")
print("=" * 42)
print(f"  True label       : {label}")
print(f"  FPGA prediction  : {fpga_pred}  {'✓' if fpga_pred==label else '✗'}")
print(f"  CPU  prediction  : {cpu_pred}  {'✓' if cpu_pred==label else '✗'}")
print("-" * 42)
print(f"  FPGA time        : {fpga_time_ms:.3f} ms")
print(f"  CPU  time        : {cpu_time_ms:.3f} ms  (avg/{N} runs)")
print(f"  Speedup          : {speedup:.2f}x  ({'FPGA faster' if speedup>1 else 'CPU faster'})")
print("=" * 42)

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()